In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [3]:
import numpy as np
import pandas as pd
import torch
from transformers import T5ForConditionalGeneration, T5Tokenizer
import wandb
from kaggle_secrets import UserSecretsClient
import warnings
warnings.filterwarnings('ignore')

options = ['A', 'B', 'C', 'D', 'E']

secrets = UserSecretsClient()
wandb.login(key=secrets.get_secret("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: varnitchourasiya27 (varnitchourasiya27-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

print("Train shape:", train.shape)
print("Test shape :", test.shape)

Train shape: (2000, 8)
Test shape : (500, 7)


In [5]:
def map_at_3(df, predict_fn):
    scores = []
    for _, row in df.iterrows():
        prediction = predict_fn(row)
        predicted_labels = prediction.split()
        correct = row['answer']
        score = 0.0
        if correct in predicted_labels:
            rank = predicted_labels.index(correct) + 1
            score = 1.0 / rank
        scores.append(score)
    return np.mean(scores)

def evaluate_and_log(model_name, predict_fn, sample_size=200):
    run = wandb.init(
        entity="varnitchourasiya27-indian-institute-of-technology-madras",
        project="23f3000843-t22026",
        name=model_name,
        config={"model": model_name, "sample_size": sample_size}
    )
    sample = train.sample(sample_size, random_state=42)
    score = map_at_3(sample, predict_fn)
    wandb.log({"MAP@3": score})
    print(f"{model_name} → Local MAP@3: {score:.4f}")
    wandb.finish()
    return score

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# flan-t5-large: Kaggle MAP@3 = 0.52784
# flan-t5-xl: Kaggle MAP@3 = 0.55652 (best)
tokenizer = T5Tokenizer.from_pretrained('google/flan-t5-xl')
model_t5 = T5ForConditionalGeneration.from_pretrained('google/flan-t5-xl')
model_t5 = model_t5.to(device)
print(f"Loaded on: {device}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loaded on: cuda


In [7]:
def predict_top3_flan(row):
    input_text = f"""Question: {row['prompt']}
A: {row['A']}
B: {row['B']}
C: {row['C']}
D: {row['D']}
E: {row['E']}
The best answer is:"""

    inputs = tokenizer(input_text, return_tensors='pt', truncation=True, max_length=1024).to(device)
    
    outputs = model_t5.generate(
        **inputs,
        max_new_tokens=5,
        num_beams=5,
        num_return_sequences=3,
        early_stopping=True
    )
    
    predictions = []
    for output in outputs:
        answer = tokenizer.decode(output, skip_special_tokens=True).strip().upper()
        if answer in options and answer not in predictions:
            predictions.append(answer)
    
    for opt in options:
        if opt not in predictions:
            predictions.append(opt)
    
    return ' '.join(predictions[:3])

In [9]:
evaluate_and_log("flan-t5-xl", predict_top3_flan, sample_size=500)

flan-t5-xl → Local MAP@3: 0.7057


MAP@3,▁
MAP@3,0.70567


np.float64(0.7056666666666667)

In [10]:
test['Prediction'] = test.apply(predict_top3_flan, axis=1)
submission = test[['id', 'Prediction']].copy()
submission.columns = ['ID', 'Prediction']
submission.to_csv('submission.csv', index=False)
print(submission.head())

   ID Prediction
0   1      D A B
1   2      B D A
2   3      D C A
3   4      E B C
4   5      A C D
